# Graph Zoo Designer — Scaling Study

**Goal:** Understand how *n_nodes* and *n_edges* (graph size and density) influence fixation probability and fixation time — providing training data for the ML models.

**Experimental axes:**
- `n_nodes`: varied across several sizes
- Edge density: `n_edges / (n_nodes*(n_nodes-1)//2)` — varied from sparse to dense at each node count
- Multiple random seeds per (n_nodes, n_edges) pair for statistical power
- Baseline graphs (complete, cycle) at each node size for reference

**Workflow:**
1. Fill in the parameter grid in Section 1
2. Build zoo in Section 2
3. Review summary in Section 3
4. Save and submit in Sections 4-5

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process 

from pathlib import Path
from moran_process import GraphZoo, PopulationGraph, ProcessLab
import pandas as pd

# Anchor to project root regardless of working directory
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {PROJECT_ROOT}")

## Section 1 — Parameter Grid

Define which (n_nodes, edge_density) combinations to study and how many random seeds per combination.

Edge density = n_edges / max_edges, where max_edges = n*(n-1)/2.  
Minimum for connectivity: n-1 edges (density ≈ 2/n for large n).  
Density 1.0 = complete graph — already added as a named baseline, so skip it in EDGE_DENSITIES.

In [ ]:
BATCH_NAME = "2026-06-10_scaling_study_6"

# DO NOT CHANGE THIS, I WILL USE THIS PART TO COMPARE THE MASSIVE BATCH WHILE OPTIMIZING THE SIMULATION

NODE_SIZES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]       # e.g. [10, 20, 31, 50, 75, 100]
EDGE_DENSITIES = [0, 0.1, 0.3, 0.5, 0.8]    # e.g. [0.1, 0.3, 0.5, 0.8]
N_SEEDS = 200         # e.g. 200
R_VALUES = [1.1]
N_REPEATS = 10_000
BATCH_SEED = None


def density_to_edges(n_nodes: int, density: float) -> int:
    '''0 is the minimum number of edges, 1 is the maximal'''
    max_edges = n_nodes * (n_nodes - 1) // 2
    min_edges = n_nodes - 1
    return min(max_edges, max(min_edges, round(density * max_edges)))


# Preview the grid
print(f"{'n_nodes':>8} {'density':>8} {'n_edges':>8} {'max_edges':>10}")
for n in NODE_SIZES:
    max_e = n * (n - 1) // 2
    for d in EDGE_DENSITIES:
        e = density_to_edges(n, d)
        print(f"{n:>8} {d:>8.2f} {e:>8} {max_e:>10}")

n_random = len(NODE_SIZES) * len(EDGE_DENSITIES) * N_SEEDS if N_SEEDS else 0
n_baselines = 2 * len(NODE_SIZES)
print(f"\nRandom graphs:  {n_random}")
print(f"Baselines:      {n_baselines}")
print(f"Grand total:    {n_random + n_baselines}")

## Section 2 — Build the Zoo

In [ ]:
zoo = GraphZoo(name=BATCH_NAME)

# Baseline graphs at each node size (theoretical fixation probabilities are known)
for n in NODE_SIZES:
    zoo.add(PopulationGraph.complete_graph(n_nodes=n))
    zoo.add(PopulationGraph.cycle_graph(n_nodes=n))
    zoo.add(PopulationGraph.star_graph(n_nodes=n))

# Random graphs across the (n_nodes, density) grid
for n in NODE_SIZES:
    for density in EDGE_DENSITIES:
        n_edges = density_to_edges(n, density)
        for seed in range(N_SEEDS):
            zoo.add(PopulationGraph.random_connected_graph(
                n_nodes=n,
                n_edges=n_edges,
                seed=seed
            ))

print(zoo)
print(f"Total graphs: {len(zoo)}")

## Section 3 — Summary
Check node/edge distribution across the zoo before committing to a run.

In [ ]:
rows = []
for g in zoo:
    n = g.graph.number_of_nodes()
    e = g.graph.number_of_edges()
    max_e = n * (n - 1) // 2
    rows.append({
        "graph_name": g.name,
        "n_nodes": n,
        "n_edges": e,
        "density": round(e / max_e, 3) if max_e > 0 else 1.0
    })

summary = pd.DataFrame(rows)
print(summary.groupby(["n_nodes", "density"])["graph_name"].count().rename("count").to_string())

## Section 4 — Save

In [ ]:
zoo_path = PROJECT_ROOT / "simulation_data" / BATCH_NAME / "zoo.pkl"
zoo.save(str(zoo_path))
print(f"Saved {len(zoo)} graphs to {zoo_path}")
categories = set(g.category for g in zoo)
print(f"Categories: {categories}")

## Section 5 — Submit HPC Jobs

In [ ]:
zoo = GraphZoo.load(str(zoo_path))
lab = ProcessLab()

lab.submit_jobs(
    zoo_path=str(zoo_path),
    r_values=R_VALUES,
    n_repeats=N_REPEATS,
    n_requested_jobs=1000,
    n_graphs=len(zoo),
    queue="gsla-cpu",
    memory="2GB",
    batch_dir=str(PROJECT_ROOT / "simulation_data" / BATCH_NAME),
    batch_name=BATCH_NAME,
    graph_types=list(categories),
    batch_seed=BATCH_SEED,
    node_sizes=NODE_SIZES,
    description="",  # fill in before submitting
    notes="",
)